### Imports & Config

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import DataFrame
from delta.tables import DeltaTable
import re
import json
from datetime import datetime
from notebookutils import mssparkutils

BRONZE_NAME = "Bronze_Layer"
SILVER_NAME = "Silver_Lakehouse"

BRONZE_EVENTS_TABLE = f"abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/2f855148-2e3c-4ca1-b8a3-6d49d05e642d/Tables/dbo/streaming_events_v3"
SILVER_PATH = "abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/190ad01e-7b58-408d-806a-4b395ab8a5bb/Tables/dbo"
WATERMARK_PATH = f"{SILVER_PATH}/_pipeline_watermark"



print(f"📂 Bronze events: {BRONZE_EVENTS_TABLE}")
print(f"📂 Silver tables: {SILVER_PATH}")
print(f"📌 Watermark:     {WATERMARK_PATH}")
print("✅ Config loaded")

StatementMeta(, b901e64f-602c-4f18-8647-c2f4a0e64db3, 3, Finished, Available, Finished, False)

📂 Bronze events: abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/2f855148-2e3c-4ca1-b8a3-6d49d05e642d/Tables/dbo/streaming_events_v3
📂 Silver tables: abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/190ad01e-7b58-408d-806a-4b395ab8a5bb/Tables/dbo
📌 Watermark:     abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/190ad01e-7b58-408d-806a-4b395ab8a5bb/Tables/dbo/_pipeline_watermark
✅ Config loaded


### Watermark Functions

In [2]:
def get_last_watermark():
    try:
        df = spark.read.format("delta").load(WATERMARK_PATH)
        last_version = df.agg(F.max("last_processed_version")).collect()[0][0]
        last_timestamp = df.agg(F.max("processed_at")).collect()[0][0]
        print(f"   📌 Last watermark: version {last_version} (processed at {last_timestamp})")
        return last_version
    except:
        print(f"   📌 No watermark found — FIRST RUN (will process everything)")
        return -1


def get_current_version(table_path):
    history = spark.sql(f"DESCRIBE HISTORY delta.`{table_path}`")
    current_version = history.agg(F.max("version")).collect()[0][0]
    return current_version


def save_watermark(version):
    watermark_data = [{
        "last_processed_version": version,
        "processed_at": datetime.now().isoformat(),
        "status": "success"
    }]
    df_watermark = spark.createDataFrame(watermark_data)
    df_watermark.write.format("delta").mode("append").save(WATERMARK_PATH)
    print(f"   📌 Watermark saved: version {version}")


print("✅ Watermark functions loaded")

StatementMeta(, b901e64f-602c-4f18-8647-c2f4a0e64db3, 4, Finished, Available, Finished, False)

✅ Watermark functions loaded


### Read ONLY New Events

In [3]:
print("=" * 60)
print("📥 READING NEW EVENTS FROM BRONZE")
print("=" * 60)

last_processed_version = get_last_watermark()
current_version = get_current_version(BRONZE_EVENTS_TABLE)

print(f"   📊 Last processed version:  {last_processed_version}")
print(f"   📊 Current table version:   {current_version}")

if current_version <= last_processed_version:
    print(f"\n   ⚠️ NO NEW EVENTS — Nothing to process!")
    mssparkutils.notebook.exit("NO_NEW_DATA")

if last_processed_version == -1:
    print(f"\n   🔄 First run — reading ALL events")
    df_raw_events = spark.read.format("delta").load(BRONZE_EVENTS_TABLE)
else:
    print(f"\n   🔄 Reading versions {last_processed_version + 1} to {current_version}")
    df_raw_events = (spark.read.format("delta")
        .option("versionAsOf", current_version)
        .load(BRONZE_EVENTS_TABLE)
    )
    df_old_events = (spark.read.format("delta")
        .option("versionAsOf", last_processed_version)
        .load(BRONZE_EVENTS_TABLE)
    )
    df_raw_events = df_raw_events.subtract(df_old_events)

total_events = df_raw_events.count()
print(f"\n   📥 New events to process: {total_events:,}")

if total_events == 0:
    print("   ⚠️ No new events after filtering — exiting")
    mssparkutils.notebook.exit("NO_NEW_DATA")

version_to_save = current_version

StatementMeta(, b901e64f-602c-4f18-8647-c2f4a0e64db3, 5, Finished, Available, Finished, False)

📥 READING NEW EVENTS FROM BRONZE
   📌 Last watermark: version 1 (processed at 2026-04-24T15:33:58.062082)
   📊 Last processed version:  1
   📊 Current table version:   2

   🔄 Reading versions 2 to 2

   📥 New events to process: 333


### Parse Payload by Event Type

In [4]:
print("=" * 60)
print("📋 PARSING EVENT PAYLOADS")
print("=" * 60)

# --- Schemas ---
order_payload_schema = StructType([
    StructField("order", StructType([
        StructField("order_id", IntegerType()),
        StructField("customer_id", IntegerType()),
        StructField("order_date", StringType()),
        StructField("total_amount", DoubleType()),
        StructField("payment_method", StringType()),
        StructField("shipping_country", StringType())
    ])),
    StructField("order_items", ArrayType(StructType([
        StructField("order_item_id", IntegerType()),
        StructField("order_id", IntegerType()),
        StructField("product_id", IntegerType()),
        StructField("quantity", IntegerType()),
        StructField("unit_price", DoubleType())
    ])))
])

new_customer_payload_schema = StructType([
    StructField("customer", StructType([
        StructField("customer_id", IntegerType()),
        StructField("name", StringType()),
        StructField("email", StringType()),
        StructField("gender", StringType()),
        StructField("start_date", StringType()),
        StructField("country", StringType()),
        StructField("end_date", StringType()),
        StructField("status", IntegerType())
    ])),
    StructField("order", StructType([
        StructField("order_id", IntegerType()),
        StructField("customer_id", IntegerType()),
        StructField("order_date", StringType()),
        StructField("total_amount", DoubleType()),
        StructField("payment_method", StringType()),
        StructField("shipping_country", StringType())
    ])),
    StructField("order_items", ArrayType(StructType([
        StructField("order_item_id", IntegerType()),
        StructField("order_id", IntegerType()),
        StructField("product_id", IntegerType()),
        StructField("quantity", IntegerType()),
        StructField("unit_price", DoubleType())
    ])))
])

customer_update_payload_schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("change_type", StringType()),
    StructField("change_date", StringType()),
    StructField("new_email", StringType()),
    StructField("new_country", StringType()),
    StructField("new_status", IntegerType()),
    StructField("end_date", StringType())
])

price_change_payload_schema = StructType([
    StructField("product_id", IntegerType()),
    StructField("old_price", DoubleType()),
    StructField("new_price", DoubleType()),
    StructField("price_change_pct", DoubleType()),
    StructField("effective_date", StringType()),
    StructField("reason", StringType())
])

# --- Split & Parse ---
df_existing_orders = (df_raw_events
    .filter(F.col("event_type") == "existing_customer_order")
    .withColumn("parsed", F.from_json(F.col("payload"), order_payload_schema))
    .select("parsed.*")
)

df_new_customer_orders = (df_raw_events
    .filter(F.col("event_type") == "new_customer_order")
    .withColumn("parsed", F.from_json(F.col("payload"), new_customer_payload_schema))
    .select("parsed.*")
)

df_customer_updates = (df_raw_events
    .filter(F.col("event_type") == "customer_update")
    .withColumn("parsed", F.from_json(F.col("payload"), customer_update_payload_schema))
    .select("parsed.*")
)

df_price_changes = (df_raw_events
    .filter(F.col("event_type") == "price_change")
    .withColumn("parsed", F.from_json(F.col("payload"), price_change_payload_schema))
    .select("parsed.*")
)

print(f"   📦 Existing orders:     {df_existing_orders.count():,}")
print(f"   👤 New customer orders: {df_new_customer_orders.count():,}")
print(f"   🔄 Customer updates:    {df_customer_updates.count():,}")
print(f"   💰 Price changes:       {df_price_changes.count():,}")
print("\n✅ All payloads parsed")

StatementMeta(, b901e64f-602c-4f18-8647-c2f4a0e64db3, 6, Finished, Available, Finished, False)

📋 PARSING EVENT PAYLOADS
   📦 Existing orders:     264
   👤 New customer orders: 49
   🔄 Customer updates:    19
   💰 Price changes:       1

✅ All payloads parsed


### Helper Functions

In [5]:
def table_exists(path: str) -> bool:
    try:
        spark.read.format("delta").load(path)
        return True
    except:
        return False


def merge_into_silver(new_df: DataFrame, table_name: str, merge_key: str, table_path: str = None):
    if table_path is None:
        table_path = f"{SILVER_PATH}/{table_name}"
    
    if not table_exists(table_path):
        new_df.write.format("delta").mode("overwrite").save(table_path)
        print(f"   ✅ {table_name}: Created new table ({new_df.count()} rows)")
        return
    
    delta_table = DeltaTable.forPath(spark, table_path)
    
    (delta_table.alias("target")
        .merge(
            new_df.alias("source"),
            f"target.{merge_key} = source.{merge_key}"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    
    final_count = spark.read.format("delta").load(table_path).count()
    print(f"   ✅ {table_name}: Merged {new_df.count()} rows (total now: {final_count:,})")


print("✅ Helper functions loaded")

StatementMeta(, b901e64f-602c-4f18-8647-c2f4a0e64db3, 7, Finished, Available, Finished, False)

✅ Helper functions loaded


### Process NEW Customers

In [6]:
print("=" * 60)
print("👤 PROCESSING: NEW CUSTOMERS")
print("=" * 60)

if df_new_customer_orders.count() > 0:
    
    df_new_customers = (df_new_customer_orders
        .select(
            F.col("customer.customer_id").cast(IntegerType()).alias("customer_id"),
            F.col("customer.name").alias("name"),
            F.lower(F.col("customer.email")).alias("email"),
            F.initcap(F.col("customer.gender")).alias("gender"),
            F.to_date(F.col("customer.start_date"), "yyyy-MM-dd").alias("start_date"),
            F.col("customer.country").alias("country"),
            F.lit(None).cast(DateType()).alias("end_date"),
            F.col("customer.status").cast(IntegerType()).alias("status")
        )
    )
    
    valid_genders = ["Male", "Female", "Other"]
    df_new_customers = df_new_customers.withColumn(
        "gender",
        F.when(F.col("gender").isin(valid_genders), F.col("gender"))
         .otherwise(F.lit("Unknown"))
    )
    
    df_new_customers = df_new_customers.filter(
        (F.col("customer_id").isNotNull()) &
        (F.col("name").isNotNull()) & (F.length(F.col("name")) > 0) &
        (F.col("email").isNotNull()) &
        (F.col("start_date").isNotNull())
    )
    
    df_new_customers = df_new_customers.dropDuplicates(["customer_id"])
    print(f"   📥 New customers to merge: {df_new_customers.count()}")
    merge_into_silver(df_new_customers, "customers", "customer_id")
else:
    print("   ⚠️ No new customer events found")

StatementMeta(, b901e64f-602c-4f18-8647-c2f4a0e64db3, 8, Finished, Available, Finished, False)

👤 PROCESSING: NEW CUSTOMERS
   📥 New customers to merge: 49
   ✅ customers: Merged 49 rows (total now: 1,048,724)


###  Process Customer Updates

In [7]:
print("=" * 60)
print("🔄 PROCESSING: CUSTOMER UPDATES")
print("=" * 60)

if df_customer_updates.count() > 0:
    
    customer_table_path = f"{SILVER_PATH}/customers"
    
    if table_exists(customer_table_path):
        delta_customers = DeltaTable.forPath(spark, customer_table_path)
        
        # --- EMAIL CHANGES ---
        df_email_changes = (df_customer_updates
            .filter(F.col("change_type") == "email_change")
            .select(
                F.col("customer_id").cast(IntegerType()).alias("customer_id"),
                F.lower(F.col("new_email")).alias("new_email")
            )
            .filter(F.col("customer_id").isNotNull() & F.col("new_email").isNotNull())
            .dropDuplicates(["customer_id"])
        )
        
        if df_email_changes.count() > 0:
            (delta_customers.alias("target")
                .merge(
                    df_email_changes.alias("source"),
                    "target.customer_id = source.customer_id"
                )
                .whenMatchedUpdate(set={"email": F.col("source.new_email")})
                .execute()
            )
            print(f"   ✅ Email changes applied: {df_email_changes.count()}")
        
        # --- COUNTRY CHANGES ---
        df_country_changes = (df_customer_updates
            .filter(F.col("change_type") == "country_change")
            .select(
                F.col("customer_id").cast(IntegerType()).alias("customer_id"),
                F.col("new_country").alias("new_country")
            )
            .filter(F.col("customer_id").isNotNull() & F.col("new_country").isNotNull())
            .dropDuplicates(["customer_id"])
        )
        
        if df_country_changes.count() > 0:
            (delta_customers.alias("target")
                .merge(
                    df_country_changes.alias("source"),
                    "target.customer_id = source.customer_id"
                )
                .whenMatchedUpdate(set={"country": F.col("source.new_country")})
                .execute()
            )
            print(f"   ✅ Country changes applied: {df_country_changes.count()}")
        
        # --- STATUS CHANGES ---
        df_status_changes = (df_customer_updates
            .filter(F.col("change_type") == "status_change")
            .select(
                F.col("customer_id").cast(IntegerType()).alias("customer_id"),
                F.col("new_status").cast(IntegerType()).alias("new_status"),
                F.to_date(F.col("end_date"), "yyyy-MM-dd").alias("new_end_date")
            )
            .filter(F.col("customer_id").isNotNull())
            .dropDuplicates(["customer_id"])
        )
        
        if df_status_changes.count() > 0:
            (delta_customers.alias("target")
                .merge(
                    df_status_changes.alias("source"),
                    "target.customer_id = source.customer_id"
                )
                .whenMatchedUpdate(set={
                    "status": F.col("source.new_status"),
                    "end_date": F.col("source.new_end_date")
                })
                .execute()
            )
            print(f"   ✅ Status changes applied: {df_status_changes.count()}")
        
        final_count = spark.read.format("delta").load(customer_table_path).count()
        print(f"   📊 Total customers in Silver: {final_count:,}")
    else:
        print("   ⚠️ Customers table doesn't exist in Silver yet")
else:
    print("   ⚠️ No customer update events found")

StatementMeta(, b901e64f-602c-4f18-8647-c2f4a0e64db3, 9, Finished, Available, Finished, False)

🔄 PROCESSING: CUSTOMER UPDATES
   ✅ Email changes applied: 3
   ✅ Country changes applied: 16
   📊 Total customers in Silver: 1,048,724


### Process New Orders

In [8]:
print("=" * 60)
print("📦 PROCESSING: NEW ORDERS")
print("=" * 60)

df_all_order_events = df_existing_orders.unionByName(df_new_customer_orders, allowMissingColumns=True)

if df_all_order_events.count() > 0:
    
    df_new_orders = (df_all_order_events
        .select(
            F.col("order.order_id").cast(IntegerType()).alias("order_id"),
            F.col("order.customer_id").cast(IntegerType()).alias("customer_id"),
            F.to_date(F.col("order.order_date"), "yyyy-MM-dd").alias("order_date"),
            F.col("order.total_amount").cast(DecimalType(12, 2)).alias("total_amount"),
            F.col("order.payment_method").alias("payment_method"),
            F.col("order.shipping_country").alias("shipping_country")
        )
    )
    
    valid_payments = ["Credit Card", "Bank Transfer", "PayPal", "Cash", "Debit Card"]
    df_new_orders = df_new_orders.withColumn(
        "payment_method",
        F.when(F.col("payment_method").isin(valid_payments), F.col("payment_method"))
         .otherwise(F.lit("Unknown"))
    )
    
    df_new_orders = df_new_orders.withColumn(
        "is_zero_amount",
        F.when(F.col("total_amount") == 0, F.lit(True)).otherwise(F.lit(False))
    )
    
    df_new_orders = df_new_orders.filter(
        (F.col("order_id").isNotNull()) &
        (F.col("customer_id").isNotNull()) &
        (F.col("order_date").isNotNull()) &
        (F.col("total_amount").isNotNull()) & (F.col("total_amount") >= 0)
    )
    
    df_new_orders = df_new_orders.dropDuplicates(["order_id"])
    print(f"   📥 New orders to merge: {df_new_orders.count()}")
    merge_into_silver(df_new_orders, "orders", "order_id")
else:
    print("   ⚠️ No order events found")

StatementMeta(, b901e64f-602c-4f18-8647-c2f4a0e64db3, 10, Finished, Available, Finished, False)

📦 PROCESSING: NEW ORDERS
   📥 New orders to merge: 313
   ✅ orders: Merged 313 rows (total now: 8,000,912)


### Process New Order Items

In [9]:
print("=" * 60)
print("📋 PROCESSING: NEW ORDER ITEMS")
print("=" * 60)

df_all_order_events = df_existing_orders.unionByName(df_new_customer_orders, allowMissingColumns=True)

if df_all_order_events.count() > 0:
    
    df_new_order_items = (df_all_order_events
        .select(F.explode(F.col("order_items")).alias("item"))
        .select(
            F.col("item.order_item_id").cast(IntegerType()).alias("order_item_id"),
            F.col("item.order_id").cast(IntegerType()).alias("order_id"),
            F.col("item.product_id").cast(IntegerType()).alias("product_id"),
            F.col("item.quantity").cast(IntegerType()).alias("quantity"),
            F.col("item.unit_price").cast(DecimalType(10, 2)).alias("unit_price")
        )
    )
    
    df_new_order_items = df_new_order_items.withColumn(
        "line_total",
        F.round(F.col("quantity") * F.col("unit_price"), 2).cast(DecimalType(12, 2))
    )
    
    df_new_order_items = df_new_order_items.filter(
        (F.col("order_item_id").isNotNull()) &
        (F.col("order_id").isNotNull()) &
        (F.col("product_id").isNotNull()) &
        (F.col("quantity").isNotNull()) & (F.col("quantity") > 0) &
        (F.col("unit_price").isNotNull()) & (F.col("unit_price") >= 0)
    )
    
    df_new_order_items = df_new_order_items.dropDuplicates(["order_item_id"])
    print(f"   📥 New order items to merge: {df_new_order_items.count()}")
    merge_into_silver(df_new_order_items, "order_items", "order_item_id")
else:
    print("   ⚠️ No order item events found")

StatementMeta(, b901e64f-602c-4f18-8647-c2f4a0e64db3, 11, Finished, Available, Finished, False)

📋 PROCESSING: NEW ORDER ITEMS
   📥 New order items to merge: 945
   ✅ order_items: Merged 945 rows (total now: 20,002,735)


### Process Price Changes

In [10]:
print("=" * 60)
print("💰 PROCESSING: PRODUCT PRICE CHANGES")
print("=" * 60)

if df_price_changes.count() > 0:
    
    products_table_path = f"{SILVER_PATH}/products"
    
    if table_exists(products_table_path):
        delta_products = DeltaTable.forPath(spark, products_table_path)
        
        df_new_prices = (df_price_changes
            .select(
                F.col("product_id").cast(IntegerType()).alias("product_id"),
                F.col("new_price").cast(DecimalType(10, 2)).alias("new_price")
            )
            .filter(
                (F.col("product_id").isNotNull()) &
                (F.col("new_price").isNotNull()) &
                (F.col("new_price") > 0)
            )
            .dropDuplicates(["product_id"])
        )
        
        if df_new_prices.count() > 0:
            df_current = spark.read.format("delta").load(products_table_path)
            df_comparison = (df_new_prices.alias("new")
                .join(df_current.select("product_id", "price").alias("old"), on="product_id", how="inner")
                .select("product_id", F.col("old.price").alias("old_price"), F.col("new.new_price").alias("new_price"))
            )
            print("   💰 Price changes:")
            df_comparison.show(truncate=False)
            
            (delta_products.alias("target")
                .merge(
                    df_new_prices.alias("source"),
                    "target.product_id = source.product_id"
                )
                .whenMatchedUpdate(set={"price": F.col("source.new_price")})
                .execute()
            )
            print(f"   ✅ Price changes applied: {df_new_prices.count()}")
    else:
        print("   ⚠️ Products table doesn't exist in Silver yet")
else:
    print("   ⚠️ No price change events found")

StatementMeta(, b901e64f-602c-4f18-8647-c2f4a0e64db3, 12, Finished, Available, Finished, False)

💰 PROCESSING: PRODUCT PRICE CHANGES
   💰 Price changes:
+----------+---------+---------+
|product_id|old_price|new_price|
+----------+---------+---------+
|15499     |446.34   |372.70   |
+----------+---------+---------+

   ✅ Price changes applied: 1


### Save Watermark

In [11]:
print("=" * 60)
print("📌 SAVING WATERMARK")
print("=" * 60)

save_watermark(version_to_save)
print(f"   ✅ Next run will start from version {version_to_save + 1}")

StatementMeta(, b901e64f-602c-4f18-8647-c2f4a0e64db3, 13, Finished, Available, Finished, False)

📌 SAVING WATERMARK
   📌 Watermark saved: version 2
   ✅ Next run will start from version 3


### Final Summary

In [12]:
print("\n")
print("╔" + "═" * 65 + "╗")
print("║" + " 📊 INCREMENTAL BRONZE → SILVER SUMMARY".center(65) + "║")
print("╠" + "═" * 65 + "╣")
print(f"║  🕐 Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}".ljust(66) + "║")
print(f"║  📥 Events processed: {total_events:,}".ljust(66) + "║")
print(f"║  📌 Watermark: version {last_processed_version} → {version_to_save}".ljust(66) + "║")
print("╠" + "═" * 65 + "╣")

for table in ["customers", "orders", "order_items", "products", "product_reviews"]:
    path = f"{SILVER_PATH}/{table}"
    try:
        df = spark.read.format("delta").load(path)
        print(f"║  📋 {table:<22} → {df.count():>14,} rows".ljust(66) + "║")
    except:
        print(f"║  ❌ {table:<22} → NOT FOUND".ljust(66) + "║")

print("╚" + "═" * 65 + "╝")
print("\n🎉 INCREMENTAL BRONZE → SILVER COMPLETE!")

StatementMeta(, b901e64f-602c-4f18-8647-c2f4a0e64db3, 14, Finished, Available, Finished, False)



╔═════════════════════════════════════════════════════════════════╗
║               📊 INCREMENTAL BRONZE → SILVER SUMMARY             ║
╠═════════════════════════════════════════════════════════════════╣
║  🕐 Completed: 2026-04-24 15:55:46                               ║
║  📥 Events processed: 333                                        ║
║  📌 Watermark: version 1 → 2                                     ║
╠═════════════════════════════════════════════════════════════════╣
║  📋 customers              →      1,048,724 rows                 ║
║  📋 orders                 →      8,000,912 rows                 ║
║  📋 order_items            →     20,002,735 rows                 ║
║  📋 products               →         20,000 rows                 ║
║  📋 product_reviews        →      4,000,000 rows                 ║
╚═════════════════════════════════════════════════════════════════╝

🎉 INCREMENTAL BRONZE → SILVER COMPLETE!
